# Checkpoint 3 - Bronze and Silver: NYC Yellow Taxi 2023

**Team T03 | ABDA Skill-Gap Capstone | Submission notebook**

This single notebook does the whole Checkpoint 3 job in order:

* **Part A - Bronze:** load the raw TLC files (and the zone lookup) into Delta tables with fixed column types.
* **Part B - Silver:** remove duplicates, add zone names and new columns, flag and remove bad trips, and write a per-month quality log.

The same code also lives in two separate notebooks in our GitHub repo (`01_bronze_load`, `02_silver_clean_and_quality`), which we use for the Databricks Job in Checkpoint 4.

**Source:** NYC Taxi & Limousine Commission (TLC), Yellow Taxi Trip Records 2023. Credit: NYC TLC.

**Before you run:** upload the Parquet files and `taxi_zone_lookup.csv` to `/Volumes/workspace/taxi/raw/` (see the step-by-step guide).

## Step 1 - Settings



In [0]:
# ===================== SETTINGS =====================
CATALOG = "workspace"   # default catalog in Databricks Free Edition
SCHEMA = "taxi"         # our own schema (think: a folder that holds our tables)
VOLUME = "raw"          # the volume (file folder) where the raw files are uploaded
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

ALL_MONTHS = [f"2023-{m:02d}" for m in range(1, 13)]
MONTHS_TO_LOAD = ALL_MONTHS     # <-- later change to: MONTHS_TO_LOAD = ALL_MONTHS

# Table names (you normally never change these)
BRONZE_TRIPS = f"{CATALOG}.{SCHEMA}.bronze_taxi_trips"
BRONZE_ZONES = f"{CATALOG}.{SCHEMA}.bronze_taxi_zones"
SILVER_TRIPS = f"{CATALOG}.{SCHEMA}.silver_taxi_trips"
QUALITY_LOG = f"{CATALOG}.{SCHEMA}.quality_log"
STAGING = f"{CATALOG}.{SCHEMA}.silver_staging"   # temporary table, dropped at the end
# ====================================================

print("Raw files folder :", VOLUME_PATH)
print("Months to load   :", MONTHS_TO_LOAD)

Raw files folder : /Volumes/workspace/taxi/raw
Months to load   : ['2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12']


## Step 2 - Imports



In [0]:
import os
from datetime import datetime
from pyspark.sql import functions as F

## Step 3 - Create the schema and the volume



In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
print("Schema and volume are ready.")

Schema and volume are ready.


## Step 4 - Check that the files are uploaded



In [0]:
expected_files = [f"yellow_tripdata_{m}.parquet" for m in MONTHS_TO_LOAD] + ["taxi_zone_lookup.csv"]

if not os.path.isdir(VOLUME_PATH):
    raise FileNotFoundError(f"Folder not found: {VOLUME_PATH}. Run Step 3 first, then upload the files to this volume.")

found = sorted(os.listdir(VOLUME_PATH))
print("Files found in the volume:")
for name in found:
    print("   ", name)

missing = [name for name in expected_files if name not in found]
if missing:
    raise FileNotFoundError(
        "These files are missing from the volume: " + ", ".join(missing)
        + ". Upload them (names must match exactly) and run this cell again."
    )
print("\nAll expected files are present.")

Files found in the volume:
    taxi_zone_lookup.csv
    yellow_tripdata_2023-01.parquet
    yellow_tripdata_2023-02.parquet
    yellow_tripdata_2023-03.parquet
    yellow_tripdata_2023-04.parquet
    yellow_tripdata_2023-05.parquet
    yellow_tripdata_2023-06.parquet
    yellow_tripdata_2023-07.parquet
    yellow_tripdata_2023-08.parquet
    yellow_tripdata_2023-09.parquet
    yellow_tripdata_2023-10.parquet
    yellow_tripdata_2023-11.parquet
    yellow_tripdata_2023-12.parquet

All expected files are present.


## Step 5 - Fix one type per column



In [0]:
COLUMN_TYPES = {
    "vendorid": "bigint",
    "tpep_pickup_datetime": "timestamp",
    "tpep_dropoff_datetime": "timestamp",
    "passenger_count": "bigint",
    "trip_distance": "double",
    "ratecodeid": "bigint",
    "store_and_fwd_flag": "string",
    "pulocationid": "bigint",
    "dolocationid": "bigint",
    "payment_type": "bigint",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "improvement_surcharge": "double",
    "total_amount": "double",
    "congestion_surcharge": "double",
    "airport_fee": "double",
}


def load_one_month(month):
    """Read one monthly Parquet file and return it with standard names and types."""
    file_name = f"yellow_tripdata_{month}.parquet"
    df = spark.read.parquet(f"{VOLUME_PATH}/{file_name}")
    df = df.toDF(*[c.lower() for c in df.columns])      # lowercase column names

    columns = []
    for name, dtype in COLUMN_TYPES.items():
        if name in df.columns:
            columns.append(F.expr(f"try_cast(`{name}` AS {dtype})").alias(name))
        else:
            columns.append(F.lit(None).cast(dtype).alias(name))   # column missing in this month

    return (
        df.select(*columns)
        .withColumn("source_file", F.lit(file_name))
        .withColumn("load_time", F.current_timestamp())
    )


print("Loader ready. Columns per month:", len(COLUMN_TYPES))

Loader ready. Columns per month: 19


## Step 6 - Load the trips into the bronze table



In [0]:
for i, month in enumerate(MONTHS_TO_LOAD):
    mode = "overwrite" if i == 0 else "append"
    writer = load_one_month(month).write.mode(mode)
    if mode == "overwrite":
        writer = writer.option("overwriteSchema", "true")
    writer.saveAsTable(BRONZE_TRIPS)
    print(f"Loaded {month}  ({mode})")

print("\nBronze trips table written:", BRONZE_TRIPS)

Loaded 2023-01  (overwrite)
Loaded 2023-02  (append)
Loaded 2023-03  (append)
Loaded 2023-04  (append)
Loaded 2023-05  (append)
Loaded 2023-06  (append)
Loaded 2023-07  (append)
Loaded 2023-08  (append)
Loaded 2023-09  (append)
Loaded 2023-10  (append)
Loaded 2023-11  (append)
Loaded 2023-12  (append)

Bronze trips table written: workspace.taxi.bronze_taxi_trips


## Step 7 - Load the zone lookup table



In [0]:
zones_raw = spark.read.option("header", True).csv(f"{VOLUME_PATH}/taxi_zone_lookup.csv")
zones_raw = zones_raw.toDF(*[c.lower() for c in zones_raw.columns])

zones_bronze = (
    zones_raw.select(
        F.expr("try_cast(locationid AS int)").alias("locationid"),
        F.col("borough"),
        F.col("zone"),
        F.col("service_zone"),
    )
    .withColumn("load_time", F.current_timestamp())
)

zones_bronze.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(BRONZE_ZONES)
print("Bronze zones table written:", BRONZE_ZONES)

Bronze zones table written: workspace.taxi.bronze_taxi_zones


## Step 8 - Check the result



In [0]:
trips = spark.table(BRONZE_TRIPS)
total_rows = trips.count()
per_file = trips.groupBy("source_file").count().orderBy("source_file")

print(f"Total bronze rows: {total_rows:,}")
display(per_file)

loaded_files = per_file.count()
assert total_rows > 0, "Bronze table is empty."
assert loaded_files == len(MONTHS_TO_LOAD), f"Expected {len(MONTHS_TO_LOAD)} files in the table but found {loaded_files}."

if "2023-01" in MONTHS_TO_LOAD:
    jan_rows = per_file.filter(F.col("source_file") == "yellow_tripdata_2023-01.parquet").first()["count"]
    print(f"January rows: {jan_rows:,}  (our plan quotes 3,066,766 from the TLC page; a small difference is fine if TLC re-issued the file)")

zone_rows = spark.table(BRONZE_ZONES).count()
print(f"Zone lookup rows: {zone_rows}  (expected about 265)")

trips.printSchema()
display(trips.limit(10))
print("Bronze checks passed.")

Total bronze rows: 38,310,226


source_file,count
yellow_tripdata_2023-01.parquet,3066766
yellow_tripdata_2023-02.parquet,2913955
yellow_tripdata_2023-03.parquet,3403766
yellow_tripdata_2023-04.parquet,3288250
yellow_tripdata_2023-05.parquet,3513649
yellow_tripdata_2023-06.parquet,3307234
yellow_tripdata_2023-07.parquet,2907108
yellow_tripdata_2023-08.parquet,2824209
yellow_tripdata_2023-09.parquet,2846722
yellow_tripdata_2023-10.parquet,3522285


January rows: 3,066,766  (our plan quotes 3,066,766 from the TLC page; a small difference is fine if TLC re-issued the file)
Zone lookup rows: 265  (expected about 265)
root
 |-- vendorid: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecodeid: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pulocationid: long (nullable = true)
 |-- dolocationid: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee:

vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,source_file,load_time
1,2023-05-01T00:33:13.000Z,2023-05-01T00:53:01.000Z,0,7.8,1,N,138,43,1,33.8,7.75,0.5,8.6,0.0,1.0,51.65,0.0,1.75,yellow_tripdata_2023-05.parquet,2026-11-07T07:25:01.070Z
1,2023-05-01T00:42:49.000Z,2023-05-01T01:11:18.000Z,2,8.1,1,N,138,262,1,35.9,10.25,0.5,9.5,0.0,1.0,57.15,2.5,1.75,yellow_tripdata_2023-05.parquet,2026-11-07T07:25:01.070Z
1,2023-05-01T00:56:34.000Z,2023-05-01T01:13:39.000Z,2,9.1,1,N,138,141,1,35.2,10.25,0.5,10.7,6.55,1.0,64.2,2.5,1.75,yellow_tripdata_2023-05.parquet,2026-11-07T07:25:01.070Z
2,2023-05-01T00:00:52.000Z,2023-05-01T00:20:12.000Z,1,8.21,1,N,138,140,1,33.1,6.0,0.5,2.24,0.0,1.0,47.09,2.5,1.75,yellow_tripdata_2023-05.parquet,2026-11-07T07:25:01.070Z
1,2023-05-01T00:05:50.000Z,2023-05-01T00:19:41.000Z,0,7.9,1,N,138,263,1,31.0,10.25,0.5,9.85,6.55,1.0,59.15,2.5,1.75,yellow_tripdata_2023-05.parquet,2026-11-07T07:25:01.070Z
1,2023-05-01T00:42:54.000Z,2023-05-01T01:04:49.000Z,0,10.4,1,N,138,246,1,42.2,10.25,0.5,8.5,6.55,1.0,69.0,2.5,1.75,yellow_tripdata_2023-05.parquet,2026-11-07T07:25:01.070Z
2,2023-05-01T00:50:34.000Z,2023-05-01T01:12:09.000Z,1,9.05,1,N,138,116,1,38.0,6.0,0.5,10.76,6.55,1.0,64.56,0.0,1.75,yellow_tripdata_2023-05.parquet,2026-11-07T07:25:01.070Z
1,2023-05-01T00:13:58.000Z,2023-05-01T00:18:10.000Z,1,0.7,1,N,161,48,1,6.5,3.5,0.5,2.85,0.0,1.0,14.35,2.5,0.0,yellow_tripdata_2023-05.parquet,2026-11-07T07:25:01.070Z
2,2023-04-30T23:48:31.000Z,2023-04-30T23:57:35.000Z,1,2.38,1,N,249,231,2,12.1,1.0,0.5,0.0,0.0,1.0,17.1,2.5,0.0,yellow_tripdata_2023-05.parquet,2026-11-07T07:25:01.070Z
2,2023-05-01T00:28:47.000Z,2023-05-01T00:39:33.000Z,1,2.92,1,N,114,230,2,14.9,1.0,0.5,0.0,0.0,1.0,19.9,2.5,0.0,yellow_tripdata_2023-05.parquet,2026-11-07T07:25:01.070Z


Bronze checks passed.


# Part B - Silver



## Step 3 - Read the bronze tables



In [0]:
RUN_ID = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

bronze = spark.table(BRONZE_TRIPS).withColumn(
    "file_month", F.regexp_extract(F.col("source_file"), r"(\d{4}-\d{2})", 1)
)
zones = spark.table(BRONZE_ZONES)

bronze_rows = bronze.count()
if bronze_rows == 0:
    raise ValueError("The bronze table is empty. Run 01_bronze_load first.")

zone_rows = zones.count()
zone_ids = zones.select("locationid").distinct().count()
assert zone_rows == zone_ids, "The zone table has repeated location IDs, which would duplicate trips in the join."

print(f"Run id           : {RUN_ID}")
print(f"Bronze rows      : {bronze_rows:,}")
print(f"Zones            : {zone_rows}")

bronze_counts = bronze.groupBy("file_month").agg(F.count(F.lit(1)).alias("rows_bronze"))

Run id           : 2026-11-07 07:25:40
Bronze rows      : 38,310,226
Zones            : 265


## Step 4 - Remove exact duplicates



In [0]:
BUSINESS_COLS = [c for c in bronze.columns if c not in ("source_file", "load_time", "file_month")]
print("Columns used to find duplicates:", len(BUSINESS_COLS))

deduped = bronze.dropDuplicates(BUSINESS_COLS)

Columns used to find duplicates: 19


## Step 5 - Add new columns and mark bad trips



In [0]:
PAYMENT_TYPES = {
    0: "Flex fare", 1: "Credit card", 2: "Cash", 3: "No charge",
    4: "Dispute", 5: "Unknown", 6: "Voided trip",
}
RATE_CODES = {
    1: "Standard rate", 2: "JFK", 3: "Newark", 4: "Nassau or Westchester",
    5: "Negotiated fare", 6: "Group ride", 99: "Unknown",
}
AIRPORT_ZONES = ["JFK Airport", "LaGuardia Airport"]


def decode(col_name, mapping, default):
    """Turn a numeric code column into a text label using a dictionary."""
    expression = None
    for key, label in mapping.items():
        condition = F.col(col_name) == F.lit(key)
        expression = F.when(condition, label) if expression is None else expression.when(condition, label)
    return expression.otherwise(default)


pickup_zones = zones.select(
    F.col("locationid").alias("_pu_id"),
    F.col("borough").alias("pu_borough"),
    F.col("zone").alias("pu_zone"),
)
dropoff_zones = zones.select(
    F.col("locationid").alias("_do_id"),
    F.col("borough").alias("do_borough"),
    F.col("zone").alias("do_zone"),
)

enriched = (
    deduped
    .join(F.broadcast(pickup_zones), F.col("pulocationid") == F.col("_pu_id"), "left").drop("_pu_id")
    .join(F.broadcast(dropoff_zones), F.col("dolocationid") == F.col("_do_id"), "left").drop("_do_id")
    .withColumn("payment_type_desc", decode("payment_type", PAYMENT_TYPES, "Other"))
    .withColumn("ratecode_desc", decode("ratecodeid", RATE_CODES, "Other"))
    .withColumn(
        "trip_minutes",
        F.round((F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0, 2),
    )
    .withColumn("fare_per_minute", F.round(F.expr("try_divide(fare_amount, trip_minutes)"), 2))
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_year", F.year("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_weekday_num", F.dayofweek("tpep_pickup_datetime"))      # 1 = Sunday ... 7 = Saturday
    .withColumn("pickup_weekday", F.date_format("tpep_pickup_datetime", "EEEE"))
    .withColumn(
        "is_airport_trip",
        F.coalesce(F.col("pu_zone").isin(AIRPORT_ZONES) | F.col("do_zone").isin(AIRPORT_ZONES), F.lit(False)),
    )
)

reject_reason = (
    F.when(
        F.col("tpep_pickup_datetime").isNull()
        | F.col("tpep_dropoff_datetime").isNull()
        | (F.col("trip_minutes") <= 0),
        "bad_times",
    )
    .when(F.col("pickup_year") != 2023, "outside_2023")
    .when(F.col("trip_minutes") > 1440, "over_24_hours")
    .when(F.col("trip_distance").isNull() | (F.col("trip_distance") <= 0), "zero_distance")
    .when(F.col("fare_amount").isNull() | (F.col("fare_amount") <= 0), "non_positive_fare")
)

flagged = enriched.withColumn("reject_reason", reject_reason)

# Final column names and order (snake_case, readable)
silver_columns = [
    F.col("vendorid").alias("vendor_id"),
    F.col("tpep_pickup_datetime").alias("pickup_datetime"),
    F.col("tpep_dropoff_datetime").alias("dropoff_datetime"),
    F.col("passenger_count"),
    F.col("trip_distance"),
    F.col("ratecodeid").alias("ratecode_id"),
    F.col("ratecode_desc"),
    F.col("store_and_fwd_flag"),
    F.col("pulocationid").alias("pu_location_id"),
    F.col("pu_borough"),
    F.col("pu_zone"),
    F.col("dolocationid").alias("do_location_id"),
    F.col("do_borough"),
    F.col("do_zone"),
    F.col("payment_type"),
    F.col("payment_type_desc"),
    F.col("fare_amount"),
    F.col("extra"),
    F.col("mta_tax"),
    F.col("tip_amount"),
    F.col("tolls_amount"),
    F.col("improvement_surcharge"),
    F.col("congestion_surcharge"),
    F.col("airport_fee"),
    F.col("total_amount"),
    F.col("trip_minutes"),
    F.col("fare_per_minute"),
    F.col("pickup_date"),
    F.col("pickup_year"),
    F.col("pickup_month"),
    F.col("pickup_hour"),
    F.col("pickup_weekday_num"),
    F.col("pickup_weekday"),
    F.col("is_airport_trip"),
    F.col("source_file"),
    F.col("file_month"),
    F.col("reject_reason"),
]
staged = flagged.select(*silver_columns)
print("Transformation defined (nothing has run yet - Spark runs it in the next step).")

Transformation defined (nothing has run yet - Spark runs it in the next step).


## Step 6 - Run the transformation once into a temporary table



In [0]:
(
    staged.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(STAGING)
)
print("Temporary table written:", STAGING)

Temporary table written: workspace.taxi.silver_staging


## Step 7 - Build the quality log



In [0]:
stage = spark.table(STAGING)


def count_reason(reason):
    return F.sum(F.when(F.col("reject_reason") == reason, 1).otherwise(0)).cast("long").alias(reason)


stage_summary = stage.groupBy("file_month").agg(
    F.count(F.lit(1)).alias("rows_after_dedupe"),
    count_reason("bad_times"),
    count_reason("outside_2023"),
    count_reason("over_24_hours"),
    count_reason("zero_distance"),
    count_reason("non_positive_fare"),
    F.sum(F.when(F.col("reject_reason").isNotNull(), 1).otherwise(0)).cast("long").alias("rows_removed"),
    F.sum(F.when(F.col("reject_reason").isNull(), 1).otherwise(0)).cast("long").alias("rows_silver"),
)

quality_log = (
    bronze_counts.join(stage_summary, "file_month", "left")
    .na.fill(0)
    .withColumn("duplicates_removed", F.col("rows_bronze") - F.col("rows_after_dedupe"))
    .withColumn("pct_kept", F.round(F.expr("try_divide(rows_silver, rows_bronze)") * 100, 2))
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("run_time", F.current_timestamp())
    .select(
        "run_id", "run_time", "file_month", "rows_bronze", "duplicates_removed", "rows_after_dedupe",
        "bad_times", "outside_2023", "over_24_hours", "zero_distance", "non_positive_fare",
        "rows_removed", "rows_silver", "pct_kept",
    )
)

quality_log.write.mode("append").saveAsTable(QUALITY_LOG)
print("Quality log saved to:", QUALITY_LOG)

Quality log saved to: workspace.taxi.quality_log


## Step 8 - Write the silver table



In [0]:
silver = (
    spark.table(STAGING)
    .filter(F.col("reject_reason").isNull())
    .drop("reject_reason")
    .withColumn("silver_load_time", F.current_timestamp())
)

silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TRIPS)
print("Silver trips table written:", SILVER_TRIPS)

Silver trips table written: workspace.taxi.silver_taxi_trips


## Step 9 - Check the result



In [0]:
log_rows = spark.table(QUALITY_LOG).filter(F.col("run_id") == RUN_ID).collect()

# Check 1: every month adds up
for row in log_rows:
    lhs = row["rows_bronze"]
    rhs = row["duplicates_removed"] + row["rows_removed"] + row["rows_silver"]
    assert lhs == rhs, f"Month {row['file_month']} does not add up: {lhs} vs {rhs}"
print("Check 1 passed: every month adds up.")

# Check 2: silver row count equals the log
silver_table = spark.table(SILVER_TRIPS)
silver_rows = silver_table.count()
expected_silver = sum(row["rows_silver"] for row in log_rows)
assert silver_rows == expected_silver, f"Silver has {silver_rows} rows but the log says {expected_silver}."
print(f"Check 2 passed: silver has {silver_rows:,} rows, matching the log.")

# Check 3 and 4: value ranges and missing values
stats = silver_table.agg(
    F.min("pickup_year").alias("min_year"),
    F.max("pickup_year").alias("max_year"),
    F.min("trip_minutes").alias("min_minutes"),
    F.max("trip_minutes").alias("max_minutes"),
    F.min("fare_amount").alias("min_fare"),
    F.min("trip_distance").alias("min_distance"),
    F.sum(F.when(F.col("pickup_datetime").isNull(), 1).otherwise(0)).alias("null_pickup"),
    F.sum(F.when(F.col("dropoff_datetime").isNull(), 1).otherwise(0)).alias("null_dropoff"),
    F.sum(F.when(F.col("fare_amount").isNull(), 1).otherwise(0)).alias("null_fare"),
).first()

assert stats["min_year"] == 2023 and stats["max_year"] == 2023, "Silver contains pickups outside 2023."
assert stats["min_minutes"] > 0 and stats["max_minutes"] <= 1440, "Trip minutes are outside 0-1440."
assert stats["min_fare"] > 0, "Silver contains a fare of 0 or less."
assert stats["min_distance"] > 0, "Silver contains a distance of 0 or less."
print("Check 3 passed: years, minutes, fares and distances are in range.")

assert stats["null_pickup"] == 0 and stats["null_dropoff"] == 0 and stats["null_fare"] == 0, "Missing values in key columns."
print("Check 4 passed: no missing values in pickup, drop-off or fare.")

print("\nAll silver checks passed.")

Check 1 passed: every month adds up.
Check 2 passed: silver has 37,185,525 rows, matching the log.
Check 3 passed: years, minutes, fares and distances are in range.
Check 4 passed: no missing values in pickup, drop-off or fare.

All silver checks passed.


## Step 10 - Look at the quality log and clean up



In [0]:
display(
    spark.table(QUALITY_LOG)
    .filter(F.col("run_id") == RUN_ID)
    .orderBy("file_month")
)

totals = spark.table(QUALITY_LOG).filter(F.col("run_id") == RUN_ID).agg(
    F.sum("rows_bronze").alias("rows_bronze"),
    F.sum("duplicates_removed").alias("duplicates_removed"),
    F.sum("bad_times").alias("bad_times"),
    F.sum("outside_2023").alias("outside_2023"),
    F.sum("over_24_hours").alias("over_24_hours"),
    F.sum("zero_distance").alias("zero_distance"),
    F.sum("non_positive_fare").alias("non_positive_fare"),
    F.sum("rows_silver").alias("rows_silver"),
)
print("TOTALS for this run (copy these numbers into the progress note):")
display(totals)

display(spark.table(SILVER_TRIPS).limit(10))

spark.sql(f"DROP TABLE IF EXISTS {STAGING}")
print("Temporary table dropped. Silver step finished.")

run_id,run_time,file_month,rows_bronze,duplicates_removed,rows_after_dedupe,bad_times,outside_2023,over_24_hours,zero_distance,non_positive_fare,rows_removed,rows_silver,pct_kept
2026-11-07 07:25:40,2026-11-07T07:26:11.382Z,2023-01,3066766,0,3066766,1121,38,25,44802,22521,68507,2998259,97.77
2026-11-07 07:25:40,2026-11-07T07:26:11.382Z,2023-02,2913955,0,2913955,1265,5,22,40184,22217,63693,2850262,97.81
2026-11-07 07:25:40,2026-11-07T07:26:11.382Z,2023-03,3403766,0,3403766,1453,11,21,47515,26633,75633,3328133,97.78
2026-11-07 07:25:40,2026-11-07T07:26:11.382Z,2023-04,3288250,0,3288250,1172,8,19,39234,27270,67703,3220547,97.94
2026-11-07 07:25:40,2026-11-07T07:26:11.382Z,2023-05,3513649,0,3513649,1170,5,7,45852,29024,76058,3437591,97.84
2026-11-07 07:25:40,2026-11-07T07:26:11.382Z,2023-06,3307234,0,3307234,1253,6,15,46474,28557,76305,3230929,97.69
2026-11-07 07:25:40,2026-11-07T07:26:11.382Z,2023-07,2907108,0,2907108,1395,4,13,48628,28422,78462,2828646,97.3
2026-11-07 07:25:40,2026-11-07T07:26:11.382Z,2023-08,2824209,0,2824209,1426,7,23,56203,28805,86464,2737745,96.94
2026-11-07 07:25:40,2026-11-07T07:26:11.382Z,2023-09,2846722,0,2846722,1066,6,16,97673,27100,125861,2720861,95.58
2026-11-07 07:25:40,2026-11-07T07:26:11.382Z,2023-10,3522285,0,3522285,1255,4,19,120595,33973,155846,3366439,95.58


TOTALS for this run (copy these numbers into the progress note):


rows_bronze,duplicates_removed,bad_times,outside_2023,over_24_hours,zero_distance,non_positive_fare,rows_silver
38310226,2,15569,104,212,761086,347728,37185525


vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,ratecode_id,ratecode_desc,store_and_fwd_flag,pu_location_id,pu_borough,pu_zone,do_location_id,do_borough,do_zone,payment_type,payment_type_desc,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,congestion_surcharge,airport_fee,total_amount,trip_minutes,fare_per_minute,pickup_date,pickup_year,pickup_month,pickup_hour,pickup_weekday_num,pickup_weekday,is_airport_trip,source_file,file_month,silver_load_time
2,2023-05-01T05:13:51.000Z,2023-05-01T05:15:32.000Z,1,0.49,1,Standard rate,N,164,Manhattan,Midtown South,100,Manhattan,Garment District,2,Cash,4.4,1.0,0.5,0.0,0.0,1.0,2.5,0.0,9.4,1.68,2.62,2023-05-01,2023,5,5,2,Monday,false,yellow_tripdata_2023-05.parquet,2023-05,2026-11-07T07:26:16.600Z
2,2023-05-01T05:48:10.000Z,2023-05-01T05:59:10.000Z,1,3.06,1,Standard rate,N,231,Manhattan,TriBeCa/Civic Center,34,Brooklyn,Brooklyn Navy Yard,1,Credit card,16.3,1.0,0.5,5.32,0.0,1.0,2.5,0.0,26.62,11.0,1.48,2023-05-01,2023,5,5,2,Monday,false,yellow_tripdata_2023-05.parquet,2023-05,2026-11-07T07:26:16.600Z
1,2023-05-01T07:24:56.000Z,2023-05-01T07:29:21.000Z,1,0.8,1,Standard rate,N,140,Manhattan,Lenox Hill East,141,Manhattan,Lenox Hill West,1,Credit card,6.5,2.5,0.5,3.1,0.0,1.0,2.5,0.0,13.6,4.42,1.47,2023-05-01,2023,5,7,2,Monday,false,yellow_tripdata_2023-05.parquet,2023-05,2026-11-07T07:26:16.600Z
1,2023-05-01T08:09:50.000Z,2023-05-01T08:29:21.000Z,1,1.8,1,Standard rate,N,239,Manhattan,Upper West Side South,236,Manhattan,Upper East Side North,1,Credit card,17.7,2.5,0.5,2.17,0.0,1.0,2.5,0.0,23.87,19.52,0.91,2023-05-01,2023,5,8,2,Monday,false,yellow_tripdata_2023-05.parquet,2023-05,2026-11-07T07:26:16.600Z
2,2023-05-01T08:43:24.000Z,2023-05-01T09:10:56.000Z,1,2.54,1,Standard rate,N,143,Manhattan,Lincoln Square West,229,Manhattan,Sutton Place/Turtle Bay North,2,Cash,23.3,0.0,0.5,0.0,0.0,1.0,2.5,0.0,27.3,27.53,0.85,2023-05-01,2023,5,8,2,Monday,false,yellow_tripdata_2023-05.parquet,2023-05,2026-11-07T07:26:16.600Z
1,2023-05-01T08:44:27.000Z,2023-05-01T08:49:32.000Z,1,0.9,1,Standard rate,N,186,Manhattan,Penn Station/Madison Sq West,234,Manhattan,Union Sq,1,Credit card,6.5,2.5,0.5,2.1,0.0,1.0,2.5,0.0,12.6,5.08,1.28,2023-05-01,2023,5,8,2,Monday,false,yellow_tripdata_2023-05.parquet,2023-05,2026-11-07T07:26:16.600Z
2,2023-05-01T08:34:18.000Z,2023-05-01T08:46:21.000Z,1,2.8,1,Standard rate,N,138,Queens,LaGuardia Airport,57,Queens,Corona,2,Cash,16.3,5.0,0.5,0.0,0.0,1.0,0.0,1.75,24.55,12.05,1.35,2023-05-01,2023,5,8,2,Monday,true,yellow_tripdata_2023-05.parquet,2023-05,2026-11-07T07:26:16.600Z
1,2023-05-01T08:42:46.000Z,2023-05-01T08:58:59.000Z,1,1.5,1,Standard rate,N,233,Manhattan,UN/Turtle Bay South,140,Manhattan,Lenox Hill East,1,Credit card,15.6,2.5,0.5,3.9,0.0,1.0,2.5,0.0,23.5,16.22,0.96,2023-05-01,2023,5,8,2,Monday,false,yellow_tripdata_2023-05.parquet,2023-05,2026-11-07T07:26:16.600Z
1,2023-05-01T09:51:13.000Z,2023-05-01T09:55:18.000Z,1,1.4,1,Standard rate,N,143,Manhattan,Lincoln Square West,246,Manhattan,West Chelsea/Hudson Yards,1,Credit card,7.9,2.5,0.5,2.35,0.0,1.0,2.5,0.0,14.25,4.08,1.94,2023-05-01,2023,5,9,2,Monday,false,yellow_tripdata_2023-05.parquet,2023-05,2026-11-07T07:26:16.600Z
2,2023-05-01T09:39:11.000Z,2023-05-01T09:46:10.000Z,1,0.94,1,Standard rate,N,229,Manhattan,Sutton Place/Turtle Bay North,161,Manhattan,Midtown Center,1,Credit card,7.9,0.0,0.5,2.38,0.0,1.0,2.5,0.0,14.28,6.98,1.13,2023-05-01,2023,5,9,2,Monday,false,yellow_tripdata_2023-05.parquet,2023-05,2026-11-07T07:26:16.600Z


Temporary table dropped. Silver step finished.


## Done

Bronze and silver are built and checked. Checkpoint 4 builds the gold tables and the dashboard from `silver_taxi_trips`.